Expects the vfi-golden-set + model dataset as input

In [1]:
import os
import cv2
import shutil
import numpy as np
import tensorflow as tf
from tensorflow import keras

# --- CONFIGURATION ---
MODEL_PATH = "/kaggle/input/vfi-golden-set-model/golden_set_septuplets/model/vfi_septuplet_epoch_31.keras"
INPUT_DATA_DIR = "/kaggle/input/vfi-golden-set-model/golden_set_septuplets/sequences"
OUTPUT_ROOT = "/kaggle/working/golden_set"

# The model strictly requires 256x256
MODEL_INPUT_SIZE = (256, 256)

# --- HELPER FUNCTIONS ---

def get_image_info(path):
    """Detects resolution and aspect ratio for the 'Up-Trip'."""
    img = cv2.imread(path)
    if img is None:
        return None
    h, w, _ = img.shape
    return {"width": w, "height": h, "shape": (h, w)}

def load_and_preprocess(path):
    img = cv2.imread(path)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # Down-Trip: Scale to model input size
    img_small = cv2.resize(img, MODEL_INPUT_SIZE, interpolation=cv2.INTER_AREA)
    return img_small.astype('float32') / 255.0

def save_and_upscale(img_array, path, target_dims):
    """Upscales model output back to original dimensions before saving."""
    # img_array is 256x256 RGB float
    # Convert to 0-255 RGB uint8
    img_uint8 = (img_array * 255.0).clip(0, 255).astype('uint8')
    
    # Up-Trip: Bicubic interpolation for better quality
    h_orig, w_orig = target_dims
    img_upscaled = cv2.resize(img_uint8, (w_orig, h_orig), interpolation=cv2.INTER_CUBIC)
    
    # Save as BGR for OpenCV
    cv2.imwrite(path, cv2.cvtColor(img_upscaled, cv2.COLOR_RGB2BGR))

def process_sequence(model, sequence_folder):
    seq_id = os.path.basename(sequence_folder)
    print(f"Processing Sequence: {seq_id}...")

    # Grab original dimensions from the first frame
    first_frame_path = os.path.join(sequence_folder, "im1.jpg")
    orig_info = get_image_info(first_frame_path)
    if not orig_info: return
    orig_dims = orig_info["shape"]

    # Load and Down-scale all 7 frames for processing
    frames_small = []
    for i in range(1, 8):
        f_path = os.path.join(sequence_folder, f"im{i}.jpg")
        img = load_and_preprocess(f_path)
        if img is None: return
        frames_small.append(img)

    # --- 1. PREDICTION TASK (Input: 1-6 -> 7) ---
    pred_input = np.concatenate(frames_small[0:6], axis=-1) 
    pred_input = np.expand_dims(pred_input, axis=0)
    pred_results = model.predict(pred_input, verbose=0)
    im7_pred_small = pred_results[0][0] 

    # --- 2. INTERPOLATION TASK (Input: 1,2,3,5,6,7 -> 4) ---
    interp_input = np.concatenate([frames_small[0], frames_small[1], frames_small[2], 
                                   frames_small[4], frames_small[5], frames_small[6]], axis=-1)
    interp_input = np.expand_dims(interp_input, axis=0)
    interp_results = model.predict(interp_input, verbose=0)
    im4_pred_small = interp_results[1][0] 

    # --- FOLDER STRUCTURE ---
    base_path = os.path.join(OUTPUT_ROOT, seq_id)
    paths = {
        "p_in": os.path.join(base_path, "prediction", "input"),
        "p_gt": os.path.join(base_path, "prediction", "gt"),
        "p_out": os.path.join(base_path, "prediction", "output"),
        "i_in": os.path.join(base_path, "interpolation", "input"),
        "i_gt": os.path.join(base_path, "interpolation", "gt"),
        "i_out": os.path.join(base_path, "interpolation", "output"),
    }
    for p in paths.values(): os.makedirs(p, exist_ok=True)

    # SAVE RESULTS (WITH UPSCALE) & COPY ORIGINAL INPUTS
    # We save predictions upscaled, and copy original frames (no resize) for GT/Input
    save_and_upscale(im7_pred_small, os.path.join(paths["p_out"], "im7_pred.jpg"), orig_dims)
    shutil.copy2(os.path.join(sequence_folder, "im7.jpg"), os.path.join(paths["p_gt"], "im7.jpg"))
    for i in range(1, 7):
        shutil.copy2(os.path.join(sequence_folder, f"im{i}.jpg"), paths["p_in"])

    save_and_upscale(im4_pred_small, os.path.join(paths["i_out"], "im4_pred.jpg"), orig_dims)
    shutil.copy2(os.path.join(sequence_folder, "im4.jpg"), os.path.join(paths["i_gt"], "im4.jpg"))
    for i in [1, 2, 3, 5, 6, 7]:
        shutil.copy2(os.path.join(sequence_folder, f"im{i}.jpg"), paths["i_in"])

def main():
    if os.path.exists(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT)
    
    print("🧠 Loading Multi-Head Septuplet Model...")
    model = keras.models.load_model(MODEL_PATH, compile=False)

    sequences = sorted([f.path for f in os.scandir(INPUT_DATA_DIR) if f.is_dir()])
    
    for seq in sequences:
        process_sequence(model, seq)

    print(f"\n✅ All results saved to {OUTPUT_ROOT} at original resolutions.")

if __name__ == "__main__":
    main()

2025-12-19 06:03:40.839135: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766124221.057198      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766124221.119716      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766124221.638617      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766124221.638657      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766124221.638660      55 computation_placer.cc:177] computation placer alr

🧠 Loading Multi-Head Septuplet Model...


I0000 00:00:1766124234.956727      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1766124234.957459      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Processing Sequence: 001...


I0000 00:00:1766124239.180089     128 service.cc:152] XLA service 0x7d95b40496a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1766124239.180126     128 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1766124239.180131     128 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1766124239.427559     128 cuda_dnn.cc:529] Loaded cuDNN version 91002
2025-12-19 06:04:02.201720: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-12-19 06:04:02.395911: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-12-19 06:04:02.873193: E external/local_xl

Processing Sequence: 002...
Processing Sequence: 003...
Processing Sequence: 004...
Processing Sequence: 005...
Processing Sequence: 006...
Processing Sequence: 007...
Processing Sequence: 008...
Processing Sequence: 009...
Processing Sequence: 010...
Processing Sequence: 011...
Processing Sequence: 012...
Processing Sequence: 013...
Processing Sequence: 014...
Processing Sequence: 015...
Processing Sequence: 016...
Processing Sequence: 017...
Processing Sequence: 018...
Processing Sequence: 019...
Processing Sequence: 020...
Processing Sequence: 021...
Processing Sequence: 022...
Processing Sequence: 023...
Processing Sequence: 024...
Processing Sequence: 025...
Processing Sequence: 026...
Processing Sequence: 027...
Processing Sequence: 028...
Processing Sequence: 029...
Processing Sequence: 030...
Processing Sequence: 031...
Processing Sequence: 032...
Processing Sequence: 033...
Processing Sequence: 034...
Processing Sequence: 035...
Processing Sequence: 036...
Processing Sequence:

In [2]:
import shutil
import os

# --- CONFIGURATION ---
# The folder we want to pack up
FOLDER_TO_ZIP = '/kaggle/working/golden_set'

# The name of the resulting file (Kaggle will add .zip automatically)
OUTPUT_ZIP_NAME = '/kaggle/working/vfi_golden_set_results'

def create_archive():
    if not os.path.exists(FOLDER_TO_ZIP):
        print(f"❌ Error: The folder {FOLDER_TO_ZIP} does not exist!")
        return

    print(f"📦 Starting to zip {FOLDER_TO_ZIP}...")
    
    # shutil.make_archive(output_filename, format, source_directory)
    # This will create: /kaggle/working/vfi_golden_set_results.zip
    output_path = shutil.make_archive(OUTPUT_ZIP_NAME, 'zip', FOLDER_TO_ZIP)
    
    # Calculate size in MB for the user
    file_size = os.path.getsize(output_path) / (1024 * 1024)
    
    print(f"✅ Success! Archive created at: {output_path}")
    print(f"📁 Total size: {file_size:.2f} MB")
    print("\n🚀 You can now find this file in your Kaggle working directory for download.")

if __name__ == "__main__":
    create_archive()

📦 Starting to zip /kaggle/working/golden_set...
✅ Success! Archive created at: /kaggle/working/vfi_golden_set_results.zip
📁 Total size: 254.10 MB

🚀 You can now find this file in your Kaggle working directory for download.
